# Sankey demo — pypowsybl scenario builder

This notebook:
1. Uses the functions from `pypowsybl_to_sankey_scenario.py` to build power-flow scenarios
2. Renders the Sankey diagram **inline** using the `@powsybl/sankey` TypeScript renderer

**One-time setup** — run this in a terminal before opening the notebook:
```bash
# 1. Activate the Python venv
source .venv/bin/activate
pip install jupyterlab ipykernel

# 2. Install JS dependencies (only needed once)
npm install

# 3. Build the Sankey JS bundle for the notebook
cd packages/sankey && npm run notebook-bundle
```
The bundle is written to `demo/dist-notebook/sankey-notebook.js`.

## 1. Imports

In [8]:
import sys, json
from pathlib import Path
from IPython.display import HTML, display

sys.path.insert(0, str(Path(".").resolve()))

import pypowsybl.network as pn
from pypowsybl_to_sankey_scenario import (
    run_dcpf,
    create_scenarios,
    export_json,
)

## 2. Load the Sankey JS bundle

This cell loads the compiled JavaScript once for the whole notebook session.
Re-run it after rebuilding the bundle (`npm run notebook-bundle`).

In [9]:
BUNDLE = Path("dist-notebook/sankey-notebook.js")
if not BUNDLE.exists():
    raise FileNotFoundError(
        f"{BUNDLE} not found.\n"
        "Run: cd packages/sankey && npm run notebook-bundle"
    )
_SANKEY_JS = BUNDLE.read_text(encoding="utf-8")
print(f"Bundle loaded ({len(_SANKEY_JS):,} bytes)")

Bundle loaded (199,184 bytes)


## 3. Helpers: `show_sankey()` and `update_sankey()`

- **`show_sankey(baseline, contingency)`** — render the widget once. Re-running it creates a new widget.
- **`update_sankey(baseline, contingency)`** — push new scenario data into the existing widget without reloading it.

In [10]:
import uuid, json
from IPython.display import display as _display, HTML, update_display as _update_display

_widget_cid    = None
_widget_width  = 820
_widget_height = 640
_LAYOUT_KEY = '_sankey_saved_layout'
_UPDATE_KEY = '_sankey_update_pending'
_ACTIVE_KEY = '_sankey_active_widget'


def _scenario_btn(cid, mode, label, btn_mode):
    cls = 'active' if mode == btn_mode else ''
    return (f'<button id="btn-{btn_mode}-{cid}" class="{cls}"'
            f' onclick="window[\'_sankey_{cid}\'].switchTo(\'{btn_mode}\')">{label}</button>')


def _render_sankey_html(baseline, contingency, width, height, mode='base'):
    cid = uuid.uuid4().hex
    baseline_json    = json.dumps(baseline)
    contingency_json = json.dumps(contingency)
    has_ctg = contingency is not None

    base_button = _scenario_btn(cid, mode, 'Baseline', 'base')
    ctg_button  = _scenario_btn(cid, mode, 'Outage',   'ctg') if has_ctg else ''

    return cid, f"""
<script>
{_SANKEY_JS}
</script>
<style>
#ctrl-{cid} {{
    display:flex; gap:8px; padding:8px 12px; flex-wrap:wrap;
    background:#f5f5f5; border:1px solid #ccc; border-bottom:none;
    font-family:monospace; align-items:center;
}}
#ctrl-{cid} button {{
    padding:4px 10px; border:1px solid #bbb; border-radius:4px;
    background:#e8e8e8; font-family:monospace; cursor:pointer;
}}
#ctrl-{cid} button.active {{ background:#2a5ab0; color:#fff; border-color:#2a5ab0; font-weight:bold; }}
#ctrl-{cid} .sep {{ color:#999; margin:0 4px; }}
#svg-{cid} {{ overflow:hidden; border:1px solid #ccc; }}
</style>
<div id="ctrl-{cid}">
  <span class="sep">Scenario:</span>
  {base_button}
  {ctg_button}
  <span class="sep">|</span>
  <button onclick="window['_r_{cid}'].stopLayout()">Stop</button>
  <button onclick="window['_r_{cid}'].startLayout()">Restart</button>
  <button id="btn-autoscale-{cid}" onclick="window['_toggleAutoscale_{cid}'](this)">Autoscale</button>
  <span class="sep">|</span>
  <button id="btn-ori-{cid}" onclick="window['_toggleOri_{cid}'](this)">vertical</button>
  <span class="sep">|</span>
  <span style="font-size:11px">stretch</span>
  <input type="range" min="-10" max="10" value="1" step="any"
         oninput="window['_r_{cid}'].setStretch(+this.value)" style="width:100px">
  <span style="font-size:11px">align</span>
  <input type="range" min="0" max="10" value="2" step="any"
         oninput="window['_r_{cid}'].setAlign(+this.value);window['_r_{cid}'].startLayout()" style="width:100px">
  <span style="font-size:11px">repulse</span>
  <input type="range" min="0" max="10" value="5" step="any"
         oninput="window['_r_{cid}'].setRepulse(+this.value);window['_r_{cid}'].startLayout()" style="width:100px">
</div>
<div id="svg-{cid}" style="width:{width}px;height:{height}px"></div>
<script>
(function() {{
  var CID        = '{cid}';
  var LAYOUT_KEY = '{_LAYOUT_KEY}';
  var UPDATE_KEY = '{_UPDATE_KEY}';
  var ACTIVE_KEY = '{_ACTIVE_KEY}';

  var base = {baseline_json};
  var ctg  = {contingency_json};
  var targetScenario = '{mode}' === 'ctg' && ctg ? ctg : base;

  // Mark this as the active widget; stale polling loops check this to self-disable.
  window[ACTIVE_KEY] = CID;

  var container = document.getElementById('svg-{cid}');
  var r = new PowsyblSankey.SankeyRenderer(container, targetScenario);
  window['_r_{cid}'] = r;

  var savedLayout = window[LAYOUT_KEY];
  if (savedLayout) {{
    r.importLayout(savedLayout);
  }}
  r.startLayout();

  var widget = {{
    renderer: r,
    base: base,
    ctg: ctg,
    currentMode: '{mode}',
    switchTo: function(mode) {{
      this.currentMode = mode;
      var sc = mode === 'ctg' && this.ctg ? this.ctg : this.base;
      this.renderer.updateScenarioFlows(sc);
      document.getElementById('btn-base-{cid}').classList.toggle('active', mode === 'base');
      var btnCtg = document.getElementById('btn-ctg-{cid}');
      if (btnCtg) btnCtg.classList.toggle('active', mode === 'ctg');
    }}
  }};
  window['_sankey_{cid}'] = widget;

  // Save layout every 500 ms so show_sankey() can restore it on re-run.
  setInterval(function() {{
    try {{ window[LAYOUT_KEY] = r.exportLayout(); }} catch(e) {{}}
  }}, 500);

  // Poll for scenario updates written by update_sankey() into the sankey-signal
  // display (same cell → same iframe → shared window).  The renderer stays alive;
  // only the data changes — same principle as Julia's update_flows!().
  setInterval(function() {{
    if (window[ACTIVE_KEY] !== CID) return;  // superseded by a newer show_sankey()
    var pending = window[UPDATE_KEY];
    if (!pending) return;
    window[UPDATE_KEY] = null;
    widget.base = pending.base;
    widget.ctg  = pending.ctg;
    var sc = widget.currentMode === 'ctg' && widget.ctg ? widget.ctg : widget.base;
    r.updateScenarioFlows(sc);
    document.getElementById('btn-base-{cid}').classList.toggle('active', widget.currentMode === 'base');
    var btnCtg = document.getElementById('btn-ctg-{cid}');
    if (btnCtg) btnCtg.classList.toggle('active', widget.currentMode === 'ctg');
  }}, 100);

  var _autoscaleOn = false;
  window['_toggleAutoscale_{cid}'] = function(btn) {{
    _autoscaleOn = !_autoscaleOn;
    r.setAutoscale(_autoscaleOn);
    btn.classList.toggle('active', _autoscaleOn);
  }};

  var _ori = 'vertical';
  window['_toggleOri_{cid}'] = function(btn) {{
    _ori = _ori === 'vertical' ? 'horizontal' : 'vertical';
    r.setOrientation(_ori);
    btn.textContent = _ori;
  }};
}})();
</script>
"""


def show_sankey(
    baseline: dict,
    contingency: dict | None = None,
    *,
    width: int = 820,
    height: int = 640,
) -> None:
    """Render the Sankey widget. Re-running creates a fresh widget."""
    global _widget_cid, _widget_width, _widget_height
    _widget_width, _widget_height = width, height
    mode = 'base'
    cid, html = _render_sankey_html(baseline, contingency, width, height, mode)
    _widget_cid = cid
    _display(HTML(html), display_id='sankey-main')
    # Signal channel: same cell → same iframe → shared window with the widget.
    # update_sankey() writes scenario JSON here; the widget polls window[UPDATE_KEY].
    _display(HTML(''), display_id='sankey-signal')


def update_sankey(baseline: dict, contingency: dict | None = None) -> None:
    """Push new scenario data to the running widget without recreating it."""
    if _widget_cid is None:
        print("No Sankey widget yet — run show_sankey() first.")
        return
    signal = json.dumps({'base': baseline, 'ctg': contingency})
    _update_display(
        HTML(f'<script>window["{_UPDATE_KEY}"] = {signal};</script>'),
        display_id='sankey-signal',
    )

## 4. Sankey widget

**Workflow:**
1. Run the **widget cell** once to render the diagram.
2. Edit `OUTAGE` in the **scenario cell** and re-run it — the diagram updates in place.

You can also click **Baseline** / **Outage** in the toolbar to switch scenarios.

In [ ]:
# Run once to list available branch IDs for use as OUTAGE values
network = pn.create_ieee14()
run_dcpf(network)
print("Lines:")
for eid in network.get_lines().index:
    print(f"  {eid}")
print("Transformers:")
for eid in network.get_2_windings_transformers().index:
    print(f"  {eid}")

Lines:
  L1-2-1
  L1-3-1
  L4-5-1
  L3-5-1
  L5-6-1
  L6-7-1
  L8-9-1
  L9-10-1
  L4-11-1
  L5-11-1
  L11-12-1
  L2-12-1
  L3-12-1
  L7-12-1
  L11-13-1
  L12-14-1
  L13-15-1
  L14-15-1
  L12-16-1
  L15-17-1
  L16-17-1
  L17-18-1
  L18-19-1
  L19-20-1
  L15-19-1
  L20-21-1
  L21-22-1
  L22-23-1
  L23-24-1
  L23-25-1
  L25-27-1
  L27-28-1
  L28-29-1
  L8-30-1
  L26-30-1
  L17-31-1
  L29-31-1
  L23-32-1
  L31-32-1
  L27-32-1
  L15-33-1
  L19-34-1
  L35-36-1
  L35-37-1
  L33-37-1
  L34-36-1
  L34-37-1
  L37-39-1
  L37-40-1
  L30-38-1
  L39-40-1
  L40-41-1
  L40-42-1
  L41-42-1
  L43-44-1
  L34-43-1
  L44-45-1
  L45-46-1
  L46-47-1
  L46-48-1
  L47-49-1
  L42-49-1
  L42-49-2
  L45-49-1
  L48-49-1
  L49-50-1
  L49-51-1
  L51-52-1
  L52-53-1
  L53-54-1
  L49-54-1
  L49-54-2
  L54-55-1
  L54-56-1
  L55-56-1
  L56-57-1
  L50-57-1
  L56-58-1
  L51-58-1
  L54-59-1
  L56-59-1
  L56-59-2
  L55-59-1
  L59-60-1
  L59-61-1
  L60-61-1
  L60-62-1
  L61-62-1
  L63-64-1
  L38-65-1
  L64-65-1
  L49-66-1
  

In [19]:
# --- Scenario cell: edit OUTAGE and re-run to update the widget below ---
# OUTAGE = "L1-2-1"  # single branch, or a list like ["L2-5-1", "L1-2-1"]
OUTAGE = ["L2-5-1", "L1-2-1", "L4-5-1"]  # single branch, or a list like ["L2-5-1", "L1-2-1"]
# OUTAGE = []
network = pn.create_ieee14()
baseline, contingency = create_scenarios(network, outage_ids=OUTAGE)
update_sankey(baseline, contingency)

/Users/benoitjeanson/vsCode/TUD/powsybl-network-viewer/packages/sankey/demo/pypowsybl_to_sankey_scenario.py:101: UserWarning: line 'L1-2-1' has no defined limit; using flow-based p_max = 2.957 pu, (flow = 1.478)
  warnings.warn(
/Users/benoitjeanson/vsCode/TUD/powsybl-network-viewer/packages/sankey/demo/pypowsybl_to_sankey_scenario.py:101: UserWarning: line 'L1-5-1' has no defined limit; using flow-based p_max = 1.423 pu, (flow = 0.712)
  warnings.warn(
/Users/benoitjeanson/vsCode/TUD/powsybl-network-viewer/packages/sankey/demo/pypowsybl_to_sankey_scenario.py:101: UserWarning: line 'L2-3-1' has no defined limit; using flow-based p_max = 1.400 pu, (flow = 0.700)
  warnings.warn(
/Users/benoitjeanson/vsCode/TUD/powsybl-network-viewer/packages/sankey/demo/pypowsybl_to_sankey_scenario.py:101: UserWarning: line 'L2-4-1' has no defined limit; using flow-based p_max = 1.103 pu, (flow = 0.552)
  warnings.warn(
/Users/benoitjeanson/vsCode/TUD/powsybl-network-viewer/packages/sankey/demo/pypowsyb

In [17]:
# --- Widget cell: run once to render the diagram ---
show_sankey(baseline, contingency)